In [ ]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

In [ ]:
import pandas as pd
from datasets import Dataset, ClassLabel
import numpy as np
import random
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from transformers import Trainer
from transformers import EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader
import gc
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import pipeline
from datasets import concatenate_datasets

In [ ]:
ZERO_SHOT_MODELS = [
    "cross-encoder/nli-MiniLM2-L6-H768",
    "typeform/distilbert-base-uncased-mnli",
    "MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli-ling-binary",
    "tasksource/deberta-small-long-nli",
    "MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33",
    "cmarkea/distilcamembert-base-nli",
    "MoritzLaurer/xtremedistil-l6-h256-zeroshot-v1.1-all-33",
    "MoritzLaurer/deberta-v3-base-zeroshot-v1"
]

LABELS = ["met palliative care needs", "unmet palliative care needs"]
LABEL_TO_NUM = {"met palliative care needs": 0, "unmet palliative care needs": 1}
LABEL_EXPANSIONS = {
    "met palliative care needs": [
        "palliative care needs are adequately met",
        "symptoms and care needs are well managed",
        "the patient is receiving appropriate palliative care",
        "care needs are sufficiently addressed"
    ],
    "unmet palliative care needs": [
        "palliative care needs are not adequately met",
        "symptoms or care needs are poorly managed",
        "the patient requires additional palliative care support",
        "care needs are not sufficiently addressed"
    ]
}

HYPOTHESIS_TEMPLATES = [
    # Generic hypothesis templates.
    "This example is about {}.",
    # Generic medical hypothesis templates. 
    "The patient's palliative care needs are {}.",
    "Overall, the patient's care needs are {}.",
    "From this note, it can be inferred that {}.",
    "This clinical note indicates that the patient's care needs are {}.",
    "Based on this note, the patient's care needs are {}.",
    # Clinical language.
    "The patient's condition suggests that {}.",
    "This note suggests that the patient is experiencing {}.",
    "The patient's current situation reflects {}.",
    # Care quality framing.
    "The patient's symptoms and care needs are {}.",
    "The patient's care is {}.",
    # Documentation style focus.
    "This note indicates a situation where {}.",
    "The note documents that {}.",
    "The record indicates that {}.",
    "The clinical documentation suggests that {}.",
]

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [ ]:
# Load the dataset
df = pd.read_csv("./dataSyntheticAll.csv")
dataset = Dataset.from_pandas(df)

# Map dataset labels to 1s or 0s.
label_map = {
    "met palliative care needs": 0,
    "unmet palliative care needs": 1
}

dataset = dataset.map(lambda x: {"label": label_map[f"{x["needs"]} palliative care needs"]})


label_feature = ClassLabel(names=["met palliative care needs", "unmet palliative care needs"])
dataset = dataset.cast_column("label", label_feature)

In [ ]:
# Split the dataset. (80/10/10)
dataset = dataset.train_test_split(test_size=0.2, seed=RANDOM_STATE, stratify_by_column="label")

train_dataset = dataset["train"]
temp_dataset = dataset["test"]

temp_split = temp_dataset.train_test_split(test_size=0.5, seed=RANDOM_STATE, stratify_by_column="label")

val_dataset = temp_split["train"]
test_dataset = temp_split["test"]

all_dataset = concatenate_datasets([train_dataset, val_dataset, test_dataset])

# Check distributions.
def check_distribution(dataset, name):
    df = dataset.to_pandas()
    counts = df["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(train_dataset, "Train")
check_distribution(val_dataset, "Validation")
check_distribution(test_dataset, "Test")
check_distribution(all_dataset, "All Data")

In [ ]:
# Make evaluation function.
def evaluate_model(preds, labels):
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
results = {}
device = 0 if torch.cuda.is_available() else -1

def run_models(dataset, hypo_temp="This clinical note indicates that {}."):
    for model_name in ZERO_SHOT_MODELS:
        print(f"\nRunning zero-shot model: {model_name}")

        classifier = pipeline(
            "zero-shot-classification",
            model=model_name,
            device=device,
        )

        texts = [ex["report"] for ex in dataset]
        true_labels = [ex["label"] for ex in dataset]

        # Flatten expanded labels.
        flat_labels = []
        label_map = {}

        for base_label, expansions in LABEL_EXPANSIONS.items():
            for exp in expansions:
                flat_labels.append(exp)
                label_map[exp] = base_label

        # Run model.
        outputs = classifier(
            texts,
            candidate_labels=flat_labels,
            hypothesis_template=hypo_temp,
            batch_size=16
        )

        preds = []

        # Aggregate scores per base label.
        for o in outputs:
            scores_by_class = {k: 0.0 for k in LABEL_EXPANSIONS.keys()}

            for label, score in zip(o["labels"], o["scores"]):
                base_label = label_map[label]
                scores_by_class[base_label] += score

            # Pick best aggregated label.
            pred = max(scores_by_class, key=scores_by_class.get)
            preds.append(pred)
        numeric_preds = [0 if pred == "met palliative care needs" else 1 for pred in preds]
        metrics = evaluate_model(numeric_preds, true_labels)

        # Inner dictionary creation.
        if model_name not in results:
            results[model_name] = {}  

        results[model_name][hypo_temp] = metrics

        # Output.
        print("----------------------------------------------")
        print("--------------- Metric Results ---------------")
        print(f"\nModel: {model_name} \nHypothesis Template: {hypo_temp}")
        for k, v in metrics.items():
            print(f"{k}: {v:.4f}")

        print("--------------- Confusion Matrix ---------------")
        print(confusion_matrix(true_labels, numeric_preds))

        print("--------------- Classification Report ---------------")
        print(classification_report(true_labels, numeric_preds))

        print("----------------------------------------------\n\n\n")

        # Free memory.
        del classifier
        torch.cuda.empty_cache()
        gc.collect()

    return results


In [ ]:
# Run all hypothesis templates.
test_results = []
for template in HYPOTHESIS_TEMPLATES:
    test_results.append(run_models(test_dataset, hypo_temp=template))

# Save in DataFrame.
rows = []

for model_name, templates in results.items():
    for hypo_template, metrics in templates.items():
        row = {"model": model_name, "hypothesis_template": hypo_template}
        row.update(metrics)
        rows.append(row)

# Save DataFrame.
test_df = pd.DataFrame(rows)
test_df.to_csv('./test_data_all_zero_shot_results.csv', index=False)

In [19]:
test_df.sort_values(by='accuracy', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
108,MoritzLaurer/deberta-v3-base-zeroshot-v1,"From this note, it can be inferred that {}.",0.868739,0.886861,0.843750,0.864769
109,MoritzLaurer/deberta-v3-base-zeroshot-v1,This clinical note indicates that the patient'...,0.863558,0.871886,0.850694,0.861160
118,MoritzLaurer/deberta-v3-base-zeroshot-v1,The record indicates that {}.,0.860104,0.887640,0.822917,0.854054
116,MoritzLaurer/deberta-v3-base-zeroshot-v1,This note indicates a situation where {}.,0.856649,0.883895,0.819444,0.850450
111,MoritzLaurer/deberta-v3-base-zeroshot-v1,The patient's condition suggests that {}.,0.854922,0.892308,0.805556,0.846715
...,...,...,...,...,...,...
84,cmarkea/distilcamembert-base-nli,The patient's symptoms and care needs are {}.,0.537133,0.642857,0.156250,0.251397
80,cmarkea/distilcamembert-base-nli,"Based on this note, the patient's care needs a...",0.530225,0.625000,0.138889,0.227273
79,cmarkea/distilcamembert-base-nli,This clinical note indicates that the patient'...,0.530225,0.611111,0.152778,0.244444
78,cmarkea/distilcamembert-base-nli,"From this note, it can be inferred that {}.",0.526770,0.620690,0.125000,0.208092


In [20]:
test_df.sort_values(by='precision', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
45,tasksource/deberta-small-long-nli,This example is about {}.,0.739206,0.925466,0.517361,0.663697
107,MoritzLaurer/deberta-v3-base-zeroshot-v1,"Overall, the patient's care needs are {}.",0.851468,0.910569,0.777778,0.838951
57,tasksource/deberta-small-long-nli,The note documents that {}.,0.803109,0.895455,0.684028,0.775591
111,MoritzLaurer/deberta-v3-base-zeroshot-v1,The patient's condition suggests that {}.,0.854922,0.892308,0.805556,0.846715
115,MoritzLaurer/deberta-v3-base-zeroshot-v1,The patient's care is {}.,0.825561,0.891213,0.739583,0.808349
...,...,...,...,...,...,...
40,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,The patient's care is {}.,0.642487,0.587852,0.940972,0.723632
35,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,"Based on this note, the patient's care needs a...",0.602763,0.558704,0.958333,0.705882
39,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,The patient's symptoms and care needs are {}.,0.571675,0.539062,0.958333,0.690000
61,MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-a...,The patient's palliative care needs are {}.,0.544041,0.521818,0.996528,0.684964


In [21]:
test_df.sort_values(by='recall', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
31,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,The patient's palliative care needs are {}.,0.519862,0.508865,0.996528,0.673709
61,MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-a...,The patient's palliative care needs are {}.,0.544041,0.521818,0.996528,0.684964
41,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,This note indicates a situation where {}.,0.658031,0.593361,0.993056,0.742857
43,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,The record indicates that {}.,0.696028,0.622271,0.989583,0.764075
44,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,The clinical documentation suggests that {}.,0.702936,0.628889,0.982639,0.766938
...,...,...,...,...,...,...
84,cmarkea/distilcamembert-base-nli,The patient's symptoms and care needs are {}.,0.537133,0.642857,0.156250,0.251397
79,cmarkea/distilcamembert-base-nli,This clinical note indicates that the patient'...,0.530225,0.611111,0.152778,0.244444
86,cmarkea/distilcamembert-base-nli,This note indicates a situation where {}.,0.538860,0.661538,0.149306,0.243626
80,cmarkea/distilcamembert-base-nli,"Based on this note, the patient's care needs a...",0.530225,0.625000,0.138889,0.227273


In [22]:
test_df.sort_values(by='f1', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
108,MoritzLaurer/deberta-v3-base-zeroshot-v1,"From this note, it can be inferred that {}.",0.868739,0.886861,0.843750,0.864769
109,MoritzLaurer/deberta-v3-base-zeroshot-v1,This clinical note indicates that the patient'...,0.863558,0.871886,0.850694,0.861160
118,MoritzLaurer/deberta-v3-base-zeroshot-v1,The record indicates that {}.,0.860104,0.887640,0.822917,0.854054
116,MoritzLaurer/deberta-v3-base-zeroshot-v1,This note indicates a situation where {}.,0.856649,0.883895,0.819444,0.850450
111,MoritzLaurer/deberta-v3-base-zeroshot-v1,The patient's condition suggests that {}.,0.854922,0.892308,0.805556,0.846715
...,...,...,...,...,...,...
84,cmarkea/distilcamembert-base-nli,The patient's symptoms and care needs are {}.,0.537133,0.642857,0.156250,0.251397
79,cmarkea/distilcamembert-base-nli,This clinical note indicates that the patient'...,0.530225,0.611111,0.152778,0.244444
86,cmarkea/distilcamembert-base-nli,This note indicates a situation where {}.,0.538860,0.661538,0.149306,0.243626
80,cmarkea/distilcamembert-base-nli,"Based on this note, the patient's care needs a...",0.530225,0.625000,0.138889,0.227273
